# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

---
Feature vector for my lane: earlier-window (days 31–60) demand and CTR, page age, length, keyword context, type and intent — one row per page.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)
lane["ctr_prev"] = lane.clicks_prev_30d / lane.impressions_prev_30d * 100
NUM = ["impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "sessions_prev_30d", "content_age_days",
       "word_count", "search_volume", "competition", "cpc"]
X = pd.concat([lane[NUM], pd.get_dummies(lane[["content_type", "main_intent"]].fillna("unknown"), dtype=int)], axis=1)
print("feature vector:", X.shape); X.head(3)

rows x cols: (30000, 44)
feature vector: (18010, 17)


,impressions_prev_30d,clicks_prev_30d,ctr_prev,sessions_prev_30d,content_age_days,word_count,search_volume,competition,cpc,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,987,13,1.317123,9,187,3221.0,10.0,0.67,2.05,0,0,1,0,0,0,1,0
1,5915,1,0.016906,2,445,2481.0,90.0,0.01,0.05,0,0,1,0,1,0,0,0
2,6089,3,0.049269,3,141,3515.0,0.0,0.00,0.00,0,0,1,0,1,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

---
Missing values: `word_count` and keyword fields are missing mostly for non-keyword content types → I add `has_` flags rather than silently filling. Categoricals (`content_type`, `main_intent`) are one-hot. All are available at the decision moment (end of days 31–60).

In [2]:
lane[NUM].isna().mean().round(3).to_frame("missing_share")

,missing_share
impressions_prev_30d,0.000
clicks_prev_30d,0.000
ctr_prev,0.000
sessions_prev_30d,0.000
content_age_days,0.000
word_count,0.290
search_volume,0.019
competition,0.019
cpc,0.019


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

---
Attack: add each suspicious column alone and check how well it predicts the label. A near-perfect score = leak. `trend_pct` and `impressions_last_30d` are label-derived (AUC ≈ 1.0); my real features stay near 0.5–0.6.

In [3]:
from sklearn.metrics import roc_auc_score
y = lane.demand_drop
sus = {"trend_pct": -lane.trend_pct, "impressions_last_30d/prev": -(lane.impressions_last_30d / lane.impressions_prev_30d),
       "avg_position (90d, overlaps)": lane.avg_position, "ctr_prev (mine)": -lane.ctr_prev, "content_age (mine)": -lane.content_age_days}
pd.Series({k: round(roc_auc_score(y, v.fillna(0)), 3) for k, v in sus.items()}, name="single-column AUC")

trend_pct                       1.000
impressions_last_30d/prev       1.000
avg_position (90d, overlaps)    0.490
ctr_prev (mine)                 0.572
content_age (mine)              0.609
Name: single-column AUC, dtype: float64

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

---
Excluded: `trend_direction`, `trend_pct`, `*_last_30d` (label window); `impressions_90d`, `clicks_90d`, `ctr`, `avg_position` (overlap label window); `days_since_last_update` (60% updated inside label window); tiers/flags; `provider_used`, `model_used`; privacy: `content_id`, `client_id` never features, never printed with names.

In [4]:
EXCLUDED = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_90d",
            "clicks_90d", "ctr", "avg_position", "days_since_last_update", "content_id", "client_id"]
assert not set(EXCLUDED) & set(X.columns); print("excluded fields absent from feature vector:", len(EXCLUDED))

excluded fields absent from feature vector: 12


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.